# 🏥 Healthcare Staff Prediction API
### Based on `Healthcare_Project.ipynb` — `GlobalLeadershipProject_v1` dataset

**Model** : `RandomForestRegressor` (n_estimators=125, max_depth=10)  
**Target** : `staff_available`  
**Features** : `Available Extra Rooms in Hospital` · `Department` · `staff_available`

---
| Step | Cell | Description |
|------|------|-------------|
| 1 | Install  | Install all required libraries |
| 2 | Train    | Build & save the model pipeline |
| 3 | Start API | Launch FastAPI server inside Jupyter |
| 4 | Test     | Run all endpoint tests inline |
| 5 | Widget   | Interactive prediction UI |


In [ ]:
# ── CELL 1 : Install dependencies ─────────────────────────────────────────
import sys
!{sys.executable} -m pip install fastapi uvicorn[standard] scikit-learn \\
    pandas numpy joblib nest_asyncio pydantic requests ipywidgets -q
print('✅ All packages installed')


In [ ]:
# ── CELL 2 : Train & save the model pipeline ────────────────────────────────
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED)

# Synthetic data matching GlobalLeadershipProject_v1 schema
N = 6000
DEPARTMENTS  = ['Gynecology','TB & Chest disease','Anesthesia',
                'Surgery','Radiotherapy','OB/GYN','Others']
DEPT_WEIGHTS = [0.30,0.18,0.15,0.14,0.12,0.07,0.04]
DEPT_BASE    = dict(zip(DEPARTMENTS,[9,7,8,10,6,8,7]))

rooms = np.random.randint(1,25,N)
depts = np.random.choice(DEPARTMENTS,N,p=DEPT_WEIGHTS)
staff = np.array([DEPT_BASE[d]+0.45*r+np.random.normal(0,1.2)
                  for d,r in zip(depts,rooms)]).clip(1).round(1)

df = pd.DataFrame({
    'Available Extra Rooms in Hospital': rooms,
    'Department': depts,
    'staff_available': staff,
})

print('Dataset shape:', df.shape)
display(df.head())

X = df[['Available Extra Rooms in Hospital','Department','staff_available']]
y = df['staff_available']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=SEED)

pre = ColumnTransformer([
    ('num','passthrough',['Available Extra Rooms in Hospital','staff_available']),
    ('cat',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),
     ['Department']),
])

pipeline = Pipeline([
    ('preprocessor', pre),
    ('model', RandomForestRegressor(
        n_estimators=125,max_depth=10,max_features='sqrt',random_state=SEED)),
])

pipeline.fit(X_train, y_train)

pred       = pipeline.predict(X_test)
train_pred = pipeline.predict(X_train)

metrics = {
    'test_r2'  : round(r2_score(y_test,pred),4),
    'test_mae' : round(mean_absolute_error(y_test,pred),4),
    'test_rmse': round(mean_squared_error(y_test,pred)**0.5,4),
    'train_r2' : round(r2_score(y_train,train_pred),4),
}

print('\n📈 Training Metrics')
for k,v in metrics.items():
    print(f'  {k:<12}: {v}')

MODEL_META = {
    'pipeline'   : pipeline,
    'departments': DEPARTMENTS,
    'features'   : ['Available Extra Rooms in Hospital','Department','staff_available'],
    'target'     : 'staff_available',
    'metrics'    : metrics,
}
joblib.dump(MODEL_META,'model_pipeline.joblib')
print('\n✅ Model saved → model_pipeline.joblib')


In [ ]:
# ── CELL 3 : Define FastAPI app & start server ───────────────────────────────
import time, threading
import nest_asyncio, uvicorn
import pandas as pd
import joblib
import requests as _req
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, field_validator
from typing import List

nest_asyncio.apply()   # allows asyncio loop inside Jupyter

META        = joblib.load('model_pipeline.joblib')
PIPELINE    = META['pipeline']
DEPARTMENTS = META['departments']
METRICS     = META['metrics']
FEATURES    = META['features']
_START      = time.time()

app = FastAPI(
    title='Healthcare Staff Prediction API',
    description='Predicts staff_available using RandomForestRegressor.',
    version='1.0.0',
)
app.add_middleware(CORSMiddleware,allow_origins=['*'],
                  allow_methods=['*'],allow_headers=['*'])

# ── Schemas ────────────────────────────────────────────────────────────────
class PredictRequest(BaseModel):
    available_extra_rooms: int   = Field(..., ge=0, le=200, examples=[10])
    department           : str   = Field(..., examples=['Gynecology'])
    staff_available      : float = Field(..., ge=0,  examples=[12.5])
    @field_validator('department')
    @classmethod
    def strip(cls,v): return v.strip()
    model_config = {'json_schema_extra':{'example':{
        'available_extra_rooms':10,'department':'Gynecology','staff_available':12.5}}}

class PredictResponse(BaseModel):
    predicted_staff_available: float
    department               : str
    available_extra_rooms    : int
    staff_available_input    : float
    status                   : str = 'success'

class BatchRequest(BaseModel):
    records: List[PredictRequest] = Field(..., max_length=50)

class BatchResponse(BaseModel):
    count      : int
    predictions: List[PredictResponse]
    status     : str = 'success'

def to_df(records):
    return pd.DataFrame([{
        'Available Extra Rooms in Hospital': r.available_extra_rooms,
        'Department'                       : r.department,
        'staff_available'                  : r.staff_available,
    } for r in records])

# ── Routes ─────────────────────────────────────────────────────────────────
@app.get('/', tags=['General'])
def root():
    return {'message':'Healthcare Staff Prediction API','docs':'/docs','health':'/health'}

@app.get('/health', tags=['General'])
def health():
    return {'status':'ok','model_loaded':True,'model_type':'RandomForestRegressor',
            'uptime_seconds':round(time.time()-_START,1),
            'supported_departments':DEPARTMENTS,'features':FEATURES,'target':'staff_available'}

@app.get('/model/info', tags=['Model'])
def model_info():
    rfr = PIPELINE.named_steps['model']
    return {'model_type':'RandomForestRegressor',
            'hyperparameters':{'n_estimators':rfr.n_estimators,'max_depth':rfr.max_depth,
                               'max_features':rfr.max_features,'random_state':rfr.random_state},
            'training_metrics':METRICS,'input_features':FEATURES,
            'target':'staff_available','supported_departments':DEPARTMENTS}

@app.post('/predict', response_model=PredictResponse, tags=['Prediction'])
def predict(req: PredictRequest):
    try:
        pred = float(PIPELINE.predict(to_df([req]))[0])
    except Exception as e:
        raise HTTPException(500, detail=str(e))
    return PredictResponse(predicted_staff_available=round(pred,2),
        department=req.department,available_extra_rooms=req.available_extra_rooms,
        staff_available_input=req.staff_available)

@app.post('/predict/batch', response_model=BatchResponse, tags=['Prediction'])
def predict_batch(req: BatchRequest):
    try:
        preds = PIPELINE.predict(to_df(req.records))
    except Exception as e:
        raise HTTPException(500, detail=str(e))
    return BatchResponse(count=len(req.records),predictions=[
        PredictResponse(predicted_staff_available=round(float(p),2),
            department=r.department,available_extra_rooms=r.available_extra_rooms,
            staff_available_input=r.staff_available)
        for r,p in zip(req.records,preds)])

# ── Start server in a background daemon thread ─────────────────────────────
def run_server():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(2)

resp = _req.get('http://127.0.0.1:8000/health')
if resp.status_code == 200:
    print('✅ API server running  →  http://127.0.0.1:8000')
    print('📖 Swagger UI         →  http://127.0.0.1:8000/docs')
else:
    print('❌ Server failed:', resp.text)


In [ ]:
# ── CELL 4 : Test all API endpoints ─────────────────────────────────────────
import requests, json
import pandas as pd

BASE = 'http://127.0.0.1:8000'
results = []

def check(label, resp, expected=200, validate=None):
    body   = resp.json()
    passed = resp.status_code == expected
    if passed and validate: passed = validate(body)
    tag = '✅ PASS' if passed else '❌ FAIL'
    print(f'  {tag}  [{resp.status_code}]  {label}')
    if not passed: print(f'         {json.dumps(body)[:160]}')
    results.append(passed)
    return body

def section(t): print(f'\n{"-"*58}\n  {t}\n{"-"*58}')

section('1. General Endpoints')
check('GET /  welcome message',
      requests.get(f'{BASE}/'), validate=lambda b: 'message' in b)
body = check('GET /health  status=ok, model_loaded=true',
      requests.get(f'{BASE}/health'),
      validate=lambda b: b.get('status')=='ok' and b.get('model_loaded'))
print(f'         uptime     : {body.get("uptime_seconds")}s')
print(f'         departments: {body.get("supported_departments")}')

section('2. Model Info')
body = check('GET /model/info  hyperparameters & metrics',
      requests.get(f'{BASE}/model/info'),
      validate=lambda b: b.get('model_type')=='RandomForestRegressor')
hp = body.get('hyperparameters',{})
mt = body.get('training_metrics',{})
print(f'         n_estimators={hp.get("n_estimators")}  max_depth={hp.get("max_depth")}')
print(f'         test_r2={mt.get("test_r2")}  test_mae={mt.get("test_mae")}')

section('3. Single Predictions')
cases = [
    ('Gynecology  rooms=10 staff=12.5',
     {'available_extra_rooms':10,'department':'Gynecology','staff_available':12.5}),
    ('Surgery     rooms=5  staff=8.0',
     {'available_extra_rooms':5,'department':'Surgery','staff_available':8.0}),
    ('TB & Chest  rooms=15 staff=14.0',
     {'available_extra_rooms':15,'department':'TB & Chest disease','staff_available':14.0}),
    ('Radiotherapy rooms=20 staff=16.0',
     {'available_extra_rooms':20,'department':'Radiotherapy','staff_available':16.0}),
    ('Unknown dept (Cardiology) graceful fallback',
     {'available_extra_rooms':8,'department':'Cardiology','staff_available':10.0}),
]
for label,payload in cases:
    r = requests.post(f'{BASE}/predict', json=payload)
    b = check(label, r, validate=lambda x: 'predicted_staff_available' in x)
    if 'predicted_staff_available' in b:
        print(f'         predicted = {b["predicted_staff_available"]}')

section('4. Validation Errors  (expect HTTP 422)')
bad = [
    ('Negative rooms',          {'available_extra_rooms':-1,'department':'Surgery','staff_available':10.0}),
    ('Missing department',      {'available_extra_rooms':5,'staff_available':10.0}),
    ('Missing staff_available', {'available_extra_rooms':5,'department':'Surgery'}),
    ('Rooms > 200',             {'available_extra_rooms':999,'department':'Surgery','staff_available':10.0}),
    ('Empty body',              {}),
]
for label,payload in bad:
    check(label, requests.post(f'{BASE}/predict',json=payload), expected=422)

section('5. Batch Prediction')
batch = {'records':[
    {'available_extra_rooms':3, 'department':'Gynecology',       'staff_available':8.0},
    {'available_extra_rooms':10,'department':'TB & Chest disease','staff_available':12.0},
    {'available_extra_rooms':7, 'department':'Surgery',           'staff_available':9.5},
    {'available_extra_rooms':18,'department':'Radiotherapy',      'staff_available':15.0},
    {'available_extra_rooms':1, 'department':'Anesthesia',        'staff_available':6.0},
    {'available_extra_rooms':22,'department':'OB/GYN',            'staff_available':18.0},
]}
body = check('6-record batch', requests.post(f'{BASE}/predict/batch',json=batch),
             validate=lambda b: b.get('count')==6)
if 'predictions' in body:
    display(pd.DataFrame([
        {'Department':p['department'],'Rooms':p['available_extra_rooms'],
         'Input Staff':p['staff_available_input'],
         'Predicted Staff':p['predicted_staff_available']}
        for p in body['predictions']]))

passed=sum(results); failed=len(results)-passed
print(f'\n{"="*58}')
print(f'  Results : {passed}/{len(results)} passed  |  {failed} failed')
if failed==0: print('  All tests passed!')
print(f'{"="*58}')


In [ ]:
# ── CELL 5 : Interactive Prediction Widget ───────────────────────────────────
import requests
import ipywidgets as widgets
from IPython.display import display, HTML

DEPARTMENTS = ['Gynecology','TB & Chest disease','Anesthesia',
               'Surgery','Radiotherapy','OB/GYN','Others']

style  = {'description_width':'180px'}
layout = widgets.Layout(width='380px')

rooms_w = widgets.IntSlider(value=10,min=0,max=50,step=1,
            description='Extra Rooms:',style=style,layout=layout)
dept_w  = widgets.Dropdown(options=DEPARTMENTS,value='Gynecology',
            description='Department:',style=style,layout=layout)
staff_w = widgets.FloatSlider(value=12.0,min=1.0,max=30.0,step=0.5,
            description='Current Staff:',style=style,layout=layout)
btn     = widgets.Button(description='Predict',button_style='primary',
            layout=widgets.Layout(width='140px',margin='10px 0'))
out     = widgets.Output()

def on_click(_):
    with out:
        out.clear_output()
        payload = {'available_extra_rooms':rooms_w.value,
                   'department':dept_w.value,
                   'staff_available':staff_w.value}
        try:
            resp = requests.post('http://127.0.0.1:8000/predict',json=payload,timeout=5)
            pred = resp.json().get('predicted_staff_available','N/A')
            color = '#2e7d32' if isinstance(pred,float) else '#c62828'
            msg = (
                '<div style="background:#e8f5e9;border-left:5px solid #4caf50;'
                'padding:15px 20px;border-radius:6px;margin-top:10px">'
                f'<b>Department :</b> {dept_w.value}<br>'
                f'<b>Extra Rooms:</b> {rooms_w.value}<br>'
                f'<b>Input Staff:</b> {staff_w.value}<br>'
                '<hr style="border:1px solid #aaa;margin:8px 0">'
                f'<b>Predicted Staff Available: '
                f'<span style="color:{color};font-size:20px">{pred}</span></b>'
                '</div>'
            )
            display(HTML(msg))
        except Exception as e:
            print('Error:', e)

btn.on_click(on_click)
display(widgets.VBox([
    widgets.HTML('<h3>Predict Staff Availability</h3>'),
    rooms_w, dept_w, staff_w, btn, out
]))


## cURL Quick Reference

```bash
# Health check
curl http://127.0.0.1:8000/health

# Single prediction
curl -X POST http://127.0.0.1:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"available_extra_rooms":10,"department":"Gynecology","staff_available":12.5}'

# Batch prediction
curl -X POST http://127.0.0.1:8000/predict/batch \
     -H "Content-Type: application/json" \
     -d '{"records":[
       {"available_extra_rooms":10,"department":"Gynecology","staff_available":12.5},
       {"available_extra_rooms":5, "department":"Surgery",   "staff_available":8.0}
     ]}'

# Open Swagger UI in browser
# http://127.0.0.1:8000/docs
```
